# **Notebook 13 – Preventing Data Leakage**

In [11]:
# Load Dataset
import pandas as pd 
df = pd.read_csv("loan_approval_dataset.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Application_ID          3000 non-null   object 
 1   Application_Date        3000 non-null   object 
 2   Age                     3000 non-null   int64  
 3   Annual_Income           2970 non-null   float64
 4   Employment_Years        3000 non-null   int64  
 5   Credit_Score            2970 non-null   float64
 6   Loan_Amount             3000 non-null   float64
 7   Loan_Term_Months        3000 non-null   int64  
 8   Existing_Loans          3000 non-null   int64  
 9   Debt_to_Income          3000 non-null   float64
 10  Employment_Type         2970 non-null   object 
 11  Education_Level         3000 non-null   object 
 12  Loan_Purpose            3000 non-null   object 
 13  Approval_Status         3000 non-null   int64  
 14  Approval_Decision_Date  3000 non-null   

## **1. What is Data Leakage?**



**Understand the Concept**
- Data leakage occurs when information that should not be available during model training is used by the model.
- It can happen when test data influences the training process.
- It can also happen when a feature contains information that is only available after the target event.
- Data leakage can produce misleadingly high model performance.
- A model affected by leakage may perform poorly on new, unseen data.

## **2. Target Leakage**



**Understand the Concept**

- Target leakage happens when a feature gives information about the target.
- This information may only be available after the target event.
- It can make the model look more accurate than it really is.
- Such features should not be used for prediction.


**Demonstrate the Concept**

**Example:**
- Use `Final_Loan_Status` to predict `Approval_Status`.

**AI/ML Usage:**
- Helps identify features that reveal the target and should not be used for prediction.

In [12]:
# relationship between feature and target
print(pd.crosstab(df["Final_Loan_Status"],df["Approval_Status"]))

Approval_Status          0    1
Final_Loan_Status              
Closed                 279    0
Disbursed                0  179
Pending_Disbursement     0   27
Rejected              2515    0


## **3. Train-Test Contamination**



**Understand the Concept**

- Train-test contamination happens when test data accidentally influences the training data.
- This can make the model perform better than it should on unseen data.
- The test dataset should always remain separate until final evaluation.



**Demonstrate the Concept**

**Example:**
- Combine training and test data before preprocessing or model training.

**AI/ML Usage:**
- Helps prevent test information from affecting the model.


In [13]:
# train-test contamination
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Approval_Status"])
y = df["Approval_Status"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# combine training and test data
X_contaminated = pd.concat([X_train, X_test])
print("Combined records:", len(X_contaminated))

Combined records: 3000


### **4. Feature Leakage**



**Understand the Concept**

- Feature leakage happens when a feature contains information that should not be available when making a prediction.
- It can make the model performance look unrealistically high.



**Demonstrate the Concept**

**Example:**

- Use `Post_Approval_Review` to predict `Approval_Status`.

**AI/ML Usage:**

- Helps identify features that should be removed before model training.

In [14]:
# Feature Leakage
X = df[["Age", "Credit_Score", "Annual_Income", "Post_Approval_Review"]]
y = df["Approval_Status"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print("Features used:")
print(X.columns.tolist())

Features used:
['Age', 'Credit_Score', 'Annual_Income', 'Post_Approval_Review']


**Explanation**

- `Post_Approval_Review` contains information from the approval process.
- Using it as a feature can cause leakage and misleading model performance.

## **5. Preprocessing Leakage**



**Understand the Concept**

- Preprocessing leakage happens when preprocessing is fitted using the entire dataset before splitting.
- This allows information from the test data to influence the training process.



**Demonstrate the Concept**

**Example:**

- Apply `StandardScaler` to the complete dataset before splitting.

**AI/ML Usage:**

- Helps understand why preprocessing should be fitted only on training data.

In [15]:
# Preprocessing Leakage
from sklearn.preprocessing import StandardScaler

X = df[["Age", "Annual_Income", "Credit_Score", "Loan_Amount"]]
y = df["Approval_Status"]

# Fill missing values
X = X.fillna(X.median())

# fit scaler before splitting
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split after preprocessing
X_train, X_test, y_train, y_test = train_test_split( X_scaled, y, test_size=0.20, random_state=42)

**Explanation**

- The scaler is fitted using the complete dataset.
- Test data information can influence the training process.

## **6. Temporal Leakage**



**Understand the Concept**

- Temporal leakage happens when future information is used to predict a past event.
- The model gets information that would not have been available at prediction time.
- This can make the model performance look unrealistically high.





**Demonstrate the Concept**

**Example:**

- Use `Approval_Decision_Date` to predict `Approval_Status`.

**AI/ML Usage:**

- Helps ensure that only information available before the prediction is used.

In [16]:
# Temporal Leakage
X = df[["Age", "Credit_Score", "Annual_Income", "Approval_Decision_Date"]]
y = df["Approval_Status"]
print("Features used:")
print(X.columns.tolist())

Features used:
['Age', 'Credit_Score', 'Annual_Income', 'Approval_Decision_Date']


**Explanation**

- `Approval_Decision_Date` is known after the application is processed.
- Using future information for prediction can cause temporal leakage.

## **7. Examples of Data Leakage**



**Understand the Concept**

- Data leakage can happen in different ways during machine learning.
- Common examples include target, feature, preprocessing, and temporal leakage.
- Identifying these examples helps prevent misleading model performance.



**Demonstrate the Concept**

**Example:**

- Check features that may contain information related to the loan approval decision.

**AI/ML Usage:**

- Helps identify and remove leakage-prone features before training the model.

In [17]:
# Check potential leakage features

leakage_features = ["Approval_Decision_Date","Post_Approval_Review", "Final_Loan_Status"]
print("Potential Leakage Features:")
print(leakage_features)



Potential Leakage Features:
['Approval_Decision_Date', 'Post_Approval_Review', 'Final_Loan_Status']


In [18]:
print("Sample Data:")
print(df[leakage_features].head())

Sample Data:
          Approval_Decision_Date Post_Approval_Review Final_Loan_Status
0  2023-01-04 00:00:00.000000000      Rejected_Review            Closed
1  2023-01-05 10:42:27.249083027      Rejected_Review          Rejected
2  2023-01-05 21:24:54.498166055      Rejected_Review          Rejected
3  2023-01-05 08:07:21.747249083      Rejected_Review          Rejected
4  2023-01-09 18:49:48.996332110      Standard_Review          Rejected


**Explanation**

- Displays features that may contain information from after the approval decision.
- Using these features for prediction can lead to data leakage.

## **8. How to Detect Leakage**



**Understand the Concept**

- Check whether any feature contains information from the future.
- Look for features that are strongly related to the target.
- Compare model performance for unusually high results.
- Check whether preprocessing was done before splitting the data.



**Demonstrate the Concept**

**Example:**

- Check the correlation between numerical features and `Approval_Status`.

**AI/ML Usage:**

- Helps identify suspicious features before training the model.

In [19]:
# Check correlation with the target
numeric_data = df.select_dtypes(include="number")

correlation = numeric_data.corr()["Approval_Status"].sort_values( ascending=False)
print(correlation)

Approval_Status     1.000000
Credit_Score        0.174866
Annual_Income       0.134969
Employment_Years    0.074358
Age                -0.005337
Existing_Loans     -0.018609
Loan_Term_Months   -0.022729
Loan_Amount        -0.056107
Debt_to_Income     -0.120728
Name: Approval_Status, dtype: float64


**Explanation**

- Shows how numerical features are related to `Approval_Status`.
- Very strong relationships can be checked further for possible data leakage.

## **9. How to Prevent Leakage**



**Understand the Concept**

- Split the dataset into training and test data before preprocessing.
- Fit scalers only on the training data.
- Use training data to calculate missing-value replacement values.
- Do not use test data to calculate mean, median, or standard deviation.
- Remove features that contain future information.
- Remove features that directly or indirectly reveal the target.
- Use only information that would be available at prediction time.
- Keep the test dataset completely separate during model development.
- Apply the same preprocessing learned from training data to validation and test data.
- Use pipelines to keep preprocessing and model training in the correct order.
- Be careful when creating new features from dates or other time-based information.
- For time-dependent problems, use chronological splitting instead of random splitting when appropriate.
- Check suspicious features that have an unusually strong relationship with the target.
- Compare model performance to identify unusually high results that may indicate leakage.
- Perform feature selection using training data only.
- Never use test-set performance to repeatedly tune the model.
- Evaluate the final model on the test data only after all preprocessing and model selection are completed.
